# RFDB → RCDB coaster specification lookup

This notebook uses the coaster names and parks already in your RFDB label table to perform a **name-directed, rate-limited** lookup on RCDB. It selects a candidate only when title and park evidence make the match clear, saves the RCDB source URL, and exports ambiguous or missing results for review.

Before a full run, review RCDB's terms and keep the output internal to the course project. This notebook does not bypass sign-in, access controls, or download protected content.

In [1]:
%pip install -q beautifulsoup4 pandas

from pathlib import Path

# Point this at the RFDB output created by the prior notebook.
RFDB_LABELS = Path("rfdb_data/rfdb_coaster_labels.csv")
OUTPUT_DIR = Path("rcdb_data")
MAX_RIDES = None       # Pilot first. Change to None for the full RFDB table.
MAX_WORKERS = 3        # Small pool: overlaps network waits without a request burst.
REQUEST_DELAY = 1.25  # Minimum spacing between request starts across all workers.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert RFDB_LABELS.exists(), f"Could not find {RFDB_LABELS.resolve()}"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import html as html_module
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import time
from threading import Lock
import unicodedata
from difflib import SequenceMatcher
from urllib.parse import urlencode, urljoin
from urllib.request import Request, urlopen

import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://rcdb.com"
last_request_at = 0.0
request_lock = Lock()

def normalized(value):
    value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9]+", "", value)

def get_html(path_or_url):
    global last_request_at
    # Serialize only request starts; downloading and parsing still overlap across workers.
    with request_lock:
        pause = REQUEST_DELAY - (time.monotonic() - last_request_at)
        if pause > 0:
            time.sleep(pause)
        last_request_at = time.monotonic()
    url = path_or_url if path_or_url.startswith("http") else urljoin(BASE_URL, path_or_url)
    request = Request(url, headers={"User-Agent": "Cal-Poly-course-project/RFDB-RCDB-lookup"})
    with urlopen(request, timeout=30) as response:
        return response.read().decode("utf-8", errors="replace"), url

def search_rcdb(name):
    query = urlencode({"ot": 2, "nc": name})
    html, source_url = get_html(f"/r.htm?{query}")
    # RCDB's compact table omits many closing </td> tags. Extract the coaster and
    # following park links from the raw, stable result-row pattern instead.
    pattern = r"<tr><td>.*?<td><a href=/(\d+)\.htm>(.*?)</a><td><a href=/(\d+)\.htm>(.*?)</a><td>"
    candidates = []
    for coaster_id, coaster_name, _park_id, park_name in re.findall(pattern, html, flags=re.DOTALL):
        clean = lambda value: html_module.unescape(re.sub(r"<.*?>", "", value)).strip()
        candidates.append({
            "rcdb_name": clean(coaster_name),
            "rcdb_park": clean(park_name),
            "rcdb_url": f"{BASE_URL}/{coaster_id}.htm",
            "search_url": source_url,
        })
    return candidates

def choose_candidate(candidates, rfdb_name, rfdb_park):
    if not candidates:
        return None, "not_found"
    name_key, park_key = normalized(rfdb_name), normalized(rfdb_park)
    scored = []
    for candidate in candidates:
        title_score = 100 if normalized(candidate["rcdb_name"]) == name_key else 100 * SequenceMatcher(None, normalized(candidate["rcdb_name"]), name_key).ratio()
        park_score = 30 if normalized(candidate["rcdb_park"]) == park_key else 30 * SequenceMatcher(None, normalized(candidate["rcdb_park"]), park_key).ratio()
        candidate = {**candidate, "match_score": round(title_score + park_score, 1)}
        scored.append(candidate)
    scored.sort(key=lambda row: row["match_score"], reverse=True)
    best = scored[0]
    tied = len(scored) > 1 and scored[1]["match_score"] == best["match_score"]
    if best["match_score"] < 120 or tied:
        return {**best, "all_candidates": scored}, "ambiguous"
    return best, "matched"

def parse_specs(rcdb_url):
    html, _ = get_html(rcdb_url)
    soup = BeautifulSoup(html, "html.parser")
    wanted = {"Length", "Height", "Drop", "Speed", "Inversions", "Vertical Angle", "Duration", "Elements"}
    specs = {}
    for table in soup.select("table"):
        # As with the search page, RCDB's compact markup is best read as a table-wide
        # sequence of visible values rather than individual table cells.
        values = list(table.stripped_strings)
        for index, value in enumerate(values[:-1]):
            if value in wanted and value not in specs:
                unit = values[index + 2] if index + 2 < len(values) and values[index + 2] in {"ft", "mph", "°"} else ""
                specs[value] = (values[index + 1] + (" " + unit if unit else "")).strip()
    return specs

In [3]:
rfdb = pd.read_csv(RFDB_LABELS)
if MAX_RIDES is not None:
    rfdb = rfdb.head(MAX_RIDES)

matched_rows, review_rows = [], []

def process_coaster(rfdb_row):
    try:
        candidates = search_rcdb(rfdb_row["ride_name"])
        selected, status = choose_candidate(candidates, rfdb_row["ride_name"], rfdb_row["park"])
        if status != "matched":
            return "review", {**rfdb_row, "rcdb_match_status": status, "rcdb_candidates": selected.get("all_candidates", []) if selected else []}
        specs = parse_specs(selected["rcdb_url"])
        return "matched", {**rfdb_row, **{f"rcdb_{key.lower().replace(' ', '_')}": value for key, value in specs.items()}, **selected, "rcdb_match_status": status}
    except Exception as error:
        return "review", {**rfdb_row, "rcdb_match_status": "error", "rcdb_candidates": str(error)}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_coaster, row) for row in rfdb.to_dict("records")]
    for position, future in enumerate(as_completed(futures), start=1):
        kind, result = future.result()
        (matched_rows if kind == "matched" else review_rows).append(result)
        print(f"[{position}/{len(rfdb)}] {result['rcdb_match_status']}: {result['ride_name']}")

matched_path = OUTPUT_DIR / "rfdb_rcdb_matched.csv"
review_path = OUTPUT_DIR / "rfdb_rcdb_review.csv"
pd.DataFrame(matched_rows).to_csv(matched_path, index=False)
pd.DataFrame(review_rows).to_csv(review_path, index=False)
print(f"\nMatched: {len(matched_rows)} | Review needed: {len(review_rows)}")
print(f"Saved: {matched_path.resolve()}")
pd.DataFrame(matched_rows).head()

[1/751] ambiguous: Zippin Pippin
[2/751] ambiguous: Roller Coaster
[3/751] ambiguous: Flight of the Himalayan Eagle
[4/751] matched: Giant Dipper
[5/751] matched: Freischütz
[6/751] matched: Abismo
[7/751] matched: Spinball Whizzer
[8/751] matched: Fridolin's verrückter Zauberexpress
[9/751] ambiguous: Family Inverted Coaster
[10/751] matched: Wicker Man
[11/751] ambiguous: Wild Mouse
[12/751] matched: Stunt Fall
[13/751] matched: Mandrill Mayhem
[14/751] matched: Wacky Worm
[15/751] matched: Wildcat
[16/751] matched: Boomerang
[17/751] ambiguous: Galaxy Spin
[18/751] ambiguous: Roller Skater
[19/751] matched: Fury
[20/751] matched: Batman: The Ride
[21/751] matched: Wicked Twister
[22/751] matched: Whistle Punk Chaser
[23/751] matched: Wilderness Run
[24/751] matched: Dragon's Fury
[25/751] not_found: Minifigure Speedway - All Stars
[26/751] not_found: Minifigure Speedway - Legends
[27/751] matched: Galactica
[28/751] matched: Hoosier Hurricane
[29/751] matched: Steel Hawg
[30/751] ma

,ride_name,park,manufacturer,valid_recording_count,miny_median,miny_iqr,maxy_median,maxy_iqr,minx_median,minx_iqr,...,rcdb_name,rcdb_park,rcdb_url,search_url,match_score,rcdb_match_status,rcdb_duration,rcdb_elements,rcdb_drop,rcdb_vertical_angle
0,Giant Dipper,Belmont Park,Prior & Church,1,0.0409,0.0,4.1443,0.0,-1.4110,0.0,...,Giant Dipper,Belmont Park,https://rcdb.com/203.htm,https://rcdb.com/r.htm?ot=2&nc=Giant+Dipper,130.0,matched,NaN,NaN,NaN,NaN
1,Freischütz,Bayern Park,Maurer Rides GmbH,1,-0.6222,0.0,4.3359,0.0,-1.5965,0.0,...,Freischütz,Bayern-Park,https://rcdb.com/9100.htm,https://rcdb.com/r.htm?ot=2&nc=Freisch%C3%BCtz,130.0,matched,0:50,LSM Launch,NaN,NaN
2,Abismo,Parque de Atracciones de Madrid,Maurer Rides GmbH,1,-1.0545,0.0,4.0489,0.0,-0.8457,0.0,...,Abismo,Parque de Atracciones de Madrid,https://rcdb.com/3185.htm,https://rcdb.com/r.htm?ot=2&nc=Abismo,130.0,matched,1:00,Vertical Chain Lift Hill,NaN,NaN
3,Spinball Whizzer,Alton Towers,Maurer Rides GmbH,1,-0.1204,0.0,2.8391,0.0,-1.2690,0.0,...,Spinball Whizzer,Alton Towers,https://rcdb.com/2472.htm,https://rcdb.com/r.htm?ot=2&nc=Spinball+Whizzer,130.0,matched,1:15,Chain Lift Hill,NaN,NaN
4,Fridolin's verrückter Zauberexpress,Fantasiana,ART Engineering GmbH,1,0.3886,0.0,2.0581,0.0,-0.9789,0.0,...,Fridolin's verrückter Zauberexpress,Fantasiana,https://rcdb.com/19040.htm,https://rcdb.com/r.htm?ot=2&nc=Fridolin%27s+ve...,130.0,matched,NaN,Tire Propelled Launch (multi-pass),NaN,NaN
